In [178]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [179]:
# drive mount
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [180]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/train.txt",sep = ';',
                 header = None,names = ['text','emotion'])

In [181]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [182]:
df.isnull().sum()

,0
text,0
emotion,0


In [183]:
unique_emotions = df["emotion"].unique()
emotion_numbers = {}
reverse_emotion_numbers = {
    v:k for k,v in emotion_numbers.items()
}
i = 0
for emotion in unique_emotions:
  emotion_numbers[emotion] = i
  i += 1

df["emotion"] = df["emotion"].map(emotion_numbers)

In [184]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


Text Cleaning Steps

In [185]:
# Lowercasing
df['text'] = df['text'].apply(lambda x : x.lower())

In [186]:
# Remove Punctuation
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [187]:
df["text"] = df['text'].apply(remove_punc)

In [188]:
# Remove Numbers
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

df['text'] = df['text'].apply(remove_numbers)

In [189]:
# Remove Emojis and Special Characters
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emojis)

In [190]:
# Remove Stopword
import nltk

In [191]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [192]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [193]:
stop_words = set(stopwords.words('english'))

negation_words = {'not', 'no', 'never'}

stop_words = stop_words - negation_words

In [194]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [195]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if i not in stop_words:
      cleaned.append(i)
  return " ".join(cleaned)

In [196]:
# Lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [197]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [198]:
df['text'] = df['text'].apply(remove)

In [199]:
df['text'] = df['text'].apply(lambda x: " ".join(
    [lemmatizer.lemmatize(word) for word in x.split()]))

In [200]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone care awake'

In [201]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [202]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2,
                                                    random_state=42, stratify=df['emotion'])

In [203]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [204]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [205]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [206]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [207]:
nb_pred = nb_model.predict(X_test_bow)

In [208]:
nb_accuracy = accuracy_score(y_test, nb_pred)
print("Naive Bayes Accuracy:", nb_accuracy)

Naive Bayes Accuracy: 0.77125


In [209]:
tfidf_vectorizer = TfidfVectorizer( max_features=10000,ngram_range=(1,2),min_df=2,max_df=0.95)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)
nb2_pred = nb2_model.predict(X_test_tfidf)
nb2_accuracy = accuracy_score(y_test, nb2_pred)
print("Naive Bayes Accuracy:", nb2_accuracy)

Naive Bayes Accuracy: 0.7296875


In [210]:
from sklearn.linear_model import LogisticRegression

In [211]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)
logistic_pred = logistic_model.predict(X_test_tfidf)
logistic_accuracy = accuracy_score(y_test, logistic_pred)
print("Logistic Regression Accuracy:", logistic_accuracy)

Logistic Regression Accuracy: 0.8659375


In [212]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(class_weight='balanced')
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)
svm_accuracy = accuracy_score(y_test, svm_pred)
print("SVM Accuracy:", svm_accuracy)

SVM Accuracy: 0.8953125


In [213]:
import joblib

In [214]:
joblib.dump(svm_model, "emotion_model.pkl")

joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']